# 02 — Multi-Dimensional Sales Analysis & ARIMA Forecasting

*Transaction-level sales intelligence: time trends, category portfolio, regional performance, discount economics, and weekly revenue forecasting*

## Executive Summary

Consumes `enhanced_df.parquet` from NB01 (hash-verified) and runs five independent analysis modules across 34,500 transactions. **2 of 4 hypotheses confirmed statistically** — category differences are real and large; discount impact on per-transaction value is real but small. Time trend and regional differences do not survive significance testing — patterns visible in charts are noise at this sample size. ARIMA(1,1,1) selected as best forecaster (MAPE = 12.21%, suitable for strategic planning).

| Output | Contents |
|---|---|
| `executive_dashboard.html` | 6-panel dashboard with statistical annotations |
| `kpi_cards.html` | 4 headline KPI cards |
| `performance_matrix.html` | Category × Region performance grid |
| `notebook02_results.pkl` | Full results dict — keys: `time_trends`, `category`, `region_payment`, `discount`, `forecasting`, `statistical_validation` |

---

> **Two analytical decisions worth noting:**
> - 555 rows (incomplete September 2025) hard-removed before time trend analysis — monthly aggregations reflect complete months only
> - Forecasting excludes 1,903 returned orders — forecasts *recognised* revenue ($5,476,538 net), not gross ordered ($5,865,293)

## 1. Environment Setup

`random_state = 42` seeded globally. Structured logging scoped to run ID. All module thresholds and config parameters externalised to `config.yaml` — nothing hardcoded downstream.

In [1]:
# Standard library
import warnings
import random
import time
from datetime import datetime
from pathlib import Path
import sys

# Third-party
import pandas as pd
import numpy as np
import plotly.graph_objects as go

# Project path resolution (supports notebooks/ or project root)
current_dir = Path.cwd()
if (current_dir.parent / 'src').exists():
    sys.path.insert(0, str(current_dir.parent / 'src'))
elif (current_dir / 'src').exists():
    sys.path.insert(0, str(current_dir / 'src'))
else:
    raise RuntimeError("Cannot find src/ directory. Run from notebooks/ or project root.")

# Core utilities
from n2a_utils import (
    get_project_root, load_config, get_config,
    setup_logger, set_run_id, generate_run_id,
    print_section_header, COLORS
)

# Run ID and logging
RUN_ID = generate_run_id()
set_run_id(RUN_ID)
logger = setup_logger(__name__)

# Configuration
PROJECT_ROOT = get_project_root()
config   = load_config(PROJECT_ROOT / 'config.yaml')
nb2_cfg  = config.get('notebook2', {})
disp_cfg = nb2_cfg.get('display', {})

pd.set_option('display.max_columns', disp_cfg.get('max_columns', None))
pd.set_option('display.max_rows',    disp_cfg.get('max_rows', 100))
pd.set_option('display.precision',   disp_cfg.get('precision', 2))
pd.set_option('display.float_format', disp_cfg.get('float_format', '{:.2f}').format)

# Reproducibility
RANDOM_SEED = config.get('notebook2', {}).get('random_state', 42)
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)
warnings.filterwarnings('ignore')

NOTEBOOK_START = time.time()

logger.info("=" * 60)
logger.info("NOTEBOOK 02: Multi-Dimensional Sales Analysis & Forecasting")
logger.info("=" * 60)
logger.info(f"Project root : {PROJECT_ROOT}")
logger.info(f"Run ID       : {RUN_ID}")
logger.info(f"Random seed  : {RANDOM_SEED}")
logger.info("=" * 60)

print(f"Setup complete | Run ID: {RUN_ID} | Seed: {RANDOM_SEED}")

[2602270033] 2026-02-27 00:33:10 - __main__ - INFO - ============================================================
[2602270033] 2026-02-27 00:33:10 - __main__ - INFO - NOTEBOOK 02: Multi-Dimensional Sales Analysis & Forecasting
[2602270033] 2026-02-27 00:33:10 - __main__ - INFO - ============================================================
[2602270033] 2026-02-27 00:33:10 - __main__ - INFO - Project root : c:\Users\Rich Justine Gambe\Documents\AAA Projects\ecommerce-analytics-project
[2602270033] 2026-02-27 00:33:10 - __main__ - INFO - Run ID       : 2602270033
[2602270033] 2026-02-27 00:33:10 - __main__ - INFO - Random seed  : 42
[2602270033] 2026-02-27 00:33:10 - __main__ - INFO - ============================================================


Setup complete | Run ID: 2602270033 | Seed: 42


## 2. Data Ingestion & Integrity Check

Loads `enhanced_df.parquet` from NB01 with hash verification — exact same dataset as NB01 output, no drift between notebooks. **34,500 rows × 17 columns**, 14/14 required columns present, total revenue $5,865,293 confirmed. Data contract intact before any analysis begins.

In [2]:
from n2b_data_loader import load_enhanced_data, print_data_summary

# Load data with validation
df = load_enhanced_data(validate=True)

# Display summary
print_data_summary(df)

[2602270033] 2026-02-27 00:33:10 - n2b_data_loader - INFO - ============================================================
[2602270033] 2026-02-27 00:33:10 - n2b_data_loader - INFO - Loading Enhanced Dataset from Notebook 01
[2602270033] 2026-02-27 00:33:10 - n2b_data_loader - INFO - ============================================================
[2602270033] 2026-02-27 00:33:10 - n2b_data_loader - INFO - Loading from: c:\Users\Rich Justine Gambe\Documents\AAA Projects\ecommerce-analytics-project\data\processed\enhanced_df.parquet
[2602270033] 2026-02-27 00:33:10 - n2b_data_loader - INFO - Loaded 34,500 rows × 17 columns
[2602270033] 2026-02-27 00:33:10 - n2b_data_loader - INFO - Date range: 2023-09-12 → 2025-09-11
[2602270033] 2026-02-27 00:33:10 - n2b_data_loader - INFO - Unique customers: 7,903
[2602270033] 2026-02-27 00:33:10 - n2b_data_loader - INFO - Total revenue: $5,865,293.00
[2602270033] 2026-02-27 00:33:10 - n2b_data_loader - INFO - Running validation checks:
[2602270033] 2026-02


                      DATASET SUMMARY                       

📊 Dimensions:
  Rows: 34,500
  Columns: 17

📅 Date Coverage:
  Start: 2023-09-12
  End: 2025-09-11
  Duration: 730 days

👥 Customers:
  Unique customers: 7,903
  Avg orders/customer: 4.37

💰 Revenue:
  Total revenue: $5,865,293.00
  Average order value: $170.01
  Median order value: $56.82

📦 Products:
  Unique products: 24,912
  Categories: 7
  Category breakdown: Home, Grocery, Electronics, Beauty, Fashion, Toys, Sports

↩️ Returns:
  Return rate: 5.52%
  Returned orders: 1,903

🗺️ Regions:
  South: 7,584 (22.0%)
  North: 7,572 (21.9%)
  East: 6,904 (20.0%)
  West: 6,808 (19.7%)
  Central: 5,632 (16.3%)


## 3. Temporal Sales Trend Analysis

Two questions: is there a statistically significant directional trend, and what is the peak seasonality pattern?

555 rows removed (hard cutoff at 2025-09-01 to exclude the incomplete final month). **24 complete months analysed** — avg monthly revenue **$240,560**, peak month **December 2024 ($278,154)**, YoY growth avg 5.23%, latest period **+14.67%**.

**Statistical result: no significant directional trend.**
Spearman ρ = 0.090, p = 0.6743. The +14.67% YoY and the flat trend test are not contradictory — Spearman tests for a monotonic direction across all 24 months; YoY measures two specific comparable windows. Month-to-month variance is high enough that no consistent direction is detectable at this time horizon. Trend-based resource allocation decisions are not supported by this data.

In [3]:
from n2c_time_trends import create_time_trends_analysis

# Run time trends analysis
time_results = create_time_trends_analysis(df, save_figures=True)

# Display figures
time_results['figures']['monthly_revenue'].show()
time_results['figures']['seasonality'].show()

[2602270033] 2026-02-27 00:33:10 - n2c_time_trends - INFO - ============================================================
[2602270033] 2026-02-27 00:33:10 - n2c_time_trends - INFO - TIME-BASED SALES TRENDS ANALYSIS
[2602270033] 2026-02-27 00:33:10 - n2c_time_trends - INFO - ============================================================
[2602270033] 2026-02-27 00:33:10 - n2c_time_trends - INFO - STEP 1: FILTERING INCOMPLETE MONTHS:
[2602270033] 2026-02-27 00:33:10 - n2c_time_trends - WARNING - Applying hard cutoff 2025-09-01 from config (removed 555 rows)
[2602270033] 2026-02-27 00:33:10 - n2c_time_trends - INFO - Note: Incomplete month data removed from analysis
[2602270033] 2026-02-27 00:33:10 - n2c_time_trends - INFO - ------------------------------------------------------------
[2602270033] 2026-02-27 00:33:10 - n2c_time_trends - INFO - STEP 2: PREPARING TIME FEATURES:
[2602270033] 2026-02-27 00:33:10 - n2c_time_trends - INFO - Adding time-based features...
[2602270033] 2026-02-27 00:3

## 4. Category Portfolio Analysis

Revenue and return rates evaluated across all 7 categories. **Pareto structure confirmed: 3 of 7 categories account for 85.7% of revenue.**

| Category | Revenue | AOV | Orders | Return Rate | Share |
|---|---|---|---|---|---|
| Electronics | $3,319,207 | $537.09 | 6,180 | 7.30% | 56.6% |
| Home | $1,077,682 | $196.41 | 5,487 | 5.65% | 18.4% |
| Sports | $629,826 | $151.00 | 4,171 | 4.94% | 10.7% |
| Fashion | $471,546 | $75.40 | 6,254 | 8.28% | 8.0% |
| Beauty | $153,019 | $37.29 | 4,103 | 3.78% | 2.6% |
| Toys | $132,014 | $31.08 | 4,247 | 4.94% | 2.3% |
| Grocery | $82,001 | $20.21 | 4,058 | 1.31% | 1.4% |

**Statistical result: category is the strongest revenue driver in the dataset.**
Kruskal-Wallis H = 18,977.30, p < 0.001, **ε² = 0.550 (large effect)**. Differences are real, large, and not a sample size artifact.

**On return rates by category:** Fashion's 8.28% and Grocery's 1.31% are statistically significant (χ² = 296.80, Cramér's V = 0.093), but the effect size is negligible. Category does not meaningfully predict returns at the transaction level — category-specific return interventions are not warranted from this data alone.

In [4]:
from n2d_category_analysis import create_category_analysis

# Run category analysis
category_results = create_category_analysis(df, save_figures=True)

# Display figures
category_results['figures']['revenue_returns'].show()
category_results['figures']['profitability'].show()
category_results['figures']['pareto'].show()

[2602270033] 2026-02-27 00:33:13 - n2d_category_analysis - INFO - ============================================================
[2602270033] 2026-02-27 00:33:13 - n2d_category_analysis - INFO - CATEGORY PERFORMANCE ANALYSIS
[2602270033] 2026-02-27 00:33:13 - n2d_category_analysis - INFO - ============================================================
[2602270033] 2026-02-27 00:33:13 - n2d_category_analysis - INFO - STEP 1: ANALYZING CATEGORY PERFORMANCE:
[2602270033] 2026-02-27 00:33:13 - n2d_category_analysis - INFO - Calculating category performance metrics...
[2602270033] 2026-02-27 00:33:13 - n2d_category_analysis - INFO - Analyzed 7 categories
[2602270033] 2026-02-27 00:33:13 - n2d_category_analysis - INFO - ------------------------------------------------------------
[2602270033] 2026-02-27 00:33:13 - n2d_category_analysis - INFO - STEP 2: PARETO ANALYSIS BY CATEGORY:
[2602270033] 2026-02-27 00:33:13 - n2d_category_analysis - INFO - Performing Pareto analysis by category...
[2602270

## 5. Regional & Payment Method Analysis

**5 regions · 6 payment methods.** South leads revenue at **$1,298,096** with the highest revenue per customer ($263.57). Average AOV across all regions: $169.94 — minimal inter-regional variation. Credit Card dominates the payment mix at 35.1%.

**Statistical result: no significant regional differences.**
Kruskal-Wallis H = 6.62, p = 0.157. The South's revenue lead is a *volume effect* — it holds 22.0% of transactions (the highest share), not a higher-value customer base. Regional segmentation strategies are not supported by this data. Resources are better directed at category-level and customer-level interventions.

In [5]:
from n2e_region_payment import create_region_payment_analysis

# Run region & payment analysis
region_results = create_region_payment_analysis(df, save_figures=True)

# Display figures
region_results['figures']['region_revenue'].show()
region_results['figures']['region_metrics'].show()
region_results['figures']['payment_dashboard'].show()

[2602270033] 2026-02-27 00:33:13 - n2e_region_payment - INFO - ============================================================
[2602270033] 2026-02-27 00:33:13 - n2e_region_payment - INFO - REGION & PAYMENT METHOD INSIGHTS
[2602270033] 2026-02-27 00:33:13 - n2e_region_payment - INFO - ============================================================
[2602270033] 2026-02-27 00:33:13 - n2e_region_payment - INFO - STEP 1: REGIONAL PERFORMANCE ANALYSIS:
[2602270033] 2026-02-27 00:33:13 - n2e_region_payment - INFO - Analyzing regional performance...
[2602270033] 2026-02-27 00:33:13 - n2e_region_payment - INFO - Analyzed 5 regions
[2602270033] 2026-02-27 00:33:13 - n2e_region_payment - INFO - Safely calculated AOV and revenue per customer
[2602270033] 2026-02-27 00:33:13 - n2e_region_payment - INFO - ------------------------------------------------------------
[2602270033] 2026-02-27 00:33:13 - n2e_region_payment - INFO - STEP 2: PAYMENT METHOD ANALYSIS:
[2602270033] 2026-02-27 00:33:13 - n2e_region

## 6. Discount Effectiveness Analysis

Two distinct questions tested independently:

**Does discounting reduce per-transaction value?**
Yes — discounted orders generate **$7.03 less per transaction (−11.7%)** (Mann-Whitney U = 140,705,141, p < 0.001, small effect). With a baseline margin of 30.07% at 0% discount, discounts primarily compress profit on individual transactions.

**Does discounting increase lifetime engagement?**
Yes — and the lift more than offsets the per-transaction loss:

| Metric | Discount Customers | Non-Discount Customers |
|---|---|---|
| Customer count | 6,846 | 1,057 |
| Median orders | 4.0 | 2.0 — **+100% frequency lift** |
| Avg lifetime revenue | $787 | $450 — **+75.1% uplift** |

**Implication:** discounted transactions earn less per order, but that loss is structurally offset by higher engagement and repeat purchasing. Discount customers are not merely price-sensitive — they are meaningfully more active. The risk is not discounting itself; it is discounts that train price sensitivity without building genuine loyalty (see NB05 fraud typology: Discount Exploiter).

In [6]:
from n2f_discount_analysis import create_discount_analysis

# Run discount analysis (now includes customer frequency test)
discount_results = create_discount_analysis(df, save_figures=True)

# Display tier comparison
discount_results['figures']['comprehensive'].show()

# Summarize frequency analysis results
fa = discount_results.get('frequency_analysis', {})
print("\nDISCOUNT FREQUENCY ANALYSIS (customer-level):\n")
print(f"Discount customers:          {fa.get('n_discount_customers', 0):,}")
print(f"Non-discount customers:      {fa.get('n_no_discount_customers', 0):,}")
print(f"Median orders — discount:    {fa.get('median_orders_discount', 0):.1f}")
print(f"Median orders — no-discount: {fa.get('median_orders_no_discount', 0):.1f}")
print(f"Frequency lift:              {fa.get('frequency_lift_pct', 0):+.1f}%")
print(f"Revenue/customer lift:       {fa.get('revenue_per_customer_lift_pct', 0):+.1f}%")
print(f"Test significant:            {fa.get('significant', False)}")
print(f"\nINTERPRETATION: {fa.get('interpretation', 'N/A')}")

[2602270033] 2026-02-27 00:33:14 - n2f_discount_analysis - INFO - ============================================================
[2602270033] 2026-02-27 00:33:14 - n2f_discount_analysis - INFO - DISCOUNT EFFECTIVENESS ANALYSIS
[2602270033] 2026-02-27 00:33:14 - n2f_discount_analysis - INFO - ============================================================
[2602270033] 2026-02-27 00:33:14 - n2f_discount_analysis - INFO - STEP 1: DISCOUNT LEVEL ANALYSIS:
[2602270033] 2026-02-27 00:33:14 - n2f_discount_analysis - INFO - Analysing discount levels...
[2602270033] 2026-02-27 00:33:14 - n2f_discount_analysis - INFO - Analysed 6 discount levels
[2602270033] 2026-02-27 00:33:14 - n2f_discount_analysis - INFO - ------------------------------------------------------------
[2602270033] 2026-02-27 00:33:14 - n2f_discount_analysis - INFO - STEP 2: CREATING COMPREHENSIVE VISUALIZATION:
[2602270033] 2026-02-27 00:33:14 - n2f_discount_analysis - INFO - Saved HTML: discount_comparison_20260227_003314.html
[26


DISCOUNT FREQUENCY ANALYSIS (customer-level):

Discount customers:          6,846
Non-discount customers:      1,057
Median orders — discount:    4.0
Median orders — no-discount: 2.0
Frequency lift:              +100.0%
Revenue/customer lift:       +75.1%
Test significant:            True

INTERPRETATION: Discount customers place significantly MORE orders per customer (median 4.0 vs 2.0, p=0.0000). Avg lifetime revenue: discount=$787 vs no-discount=$450 (+75.1%).


## 7. ARIMA Revenue Forecasting

Weekly revenue time series (returns excluded, net revenue $5,476,538). 105-week series, 84-week train / 21-week test split — chronological, no leakage. Six ARIMA configurations evaluated:

| Model | MAPE | AIC |
|---|---|---|
| **ARIMA(1,1,1)** | **12.21%** | **1734.3** |
| ARIMA(1,1,2) | 12.79% | 1739.8 |
| ARIMA(0,1,1) | 13.30% | 1734.8 |
| ARIMA(2,1,1) | 15.12% | 1740.2 |
| ARIMA(1,1,0) | 16.40% | 1739.6 |
| ARIMA(2,1,2) | 16.41% | 1741.6 |

**Selected: ARIMA(1,1,1)** — lowest MAPE *and* lowest AIC, dominant on both predictive accuracy and model parsimony.

MAE = $5,280/week · RMSE = $7,496/week · Weekly avg revenue: $52,158 (monthly equivalent: $226,792). Rolling 13-week trend: **−4.8%** (recent softening). This does not contradict the +14.67% YoY figure — the rolling window compares the last 13 weeks to the immediately prior 13 weeks, not the same period last year.

**MAPE of 12.21% is rated *Good* (< 20%)** — suitable for budget planning and strategic allocation, not operational scheduling.

In [7]:
from n2i_forecasting import create_forecasting_analysis, interpret_forecast_metrics, REVENUE_GROWTH_WEEKS

forecast_results = create_forecasting_analysis(df, save_figures=True)
forecast_results['figures']['forecast'].show()

fc_m     = forecast_results['metrics']
verdicts = interpret_forecast_metrics(fc_m)

print("\nFORECAST METRICS:")
print(f"Best model         : {fc_m['best_model_name']}")
print(f"MAPE               : {fc_m['best_model_mape']:.2f}%  → {verdicts['mape_verdict']}")
print(f"MAE                : ${fc_m['best_model_mae']:,.2f}/week")
print(f"RMSE               : ${fc_m['best_model_rmse']:,.2f}/week")
print(f"Revenue trend      : {fc_m['revenue_growth']:+.1f}%  ({REVENUE_GROWTH_WEEKS}wk rolling)")
print(f"Verdict            : {verdicts['trend_verdict']}")
if verdicts['trend_note']:
    print(f"Note               : {verdicts['trend_note']}")
print(f"Weekly avg revenue : ${fc_m['weekly_avg_revenue']:,.0f}")
print(f"Monthly equivalent : ${fc_m['monthly_equivalent']:,.0f}")

[2602270033] 2026-02-27 00:33:15 - n2i_forecasting - INFO - ============================================================
[2602270033] 2026-02-27 00:33:15 - n2i_forecasting - INFO - TIME SERIES FORECASTING ANALYSIS
[2602270033] 2026-02-27 00:33:15 - n2i_forecasting - INFO - ============================================================
[2602270033] 2026-02-27 00:33:15 - n2i_forecasting - INFO - PREPARING WEEKLY SALES DATA:
[2602270033] 2026-02-27 00:33:15 - n2i_forecasting - INFO - Preparing weekly sales data...
[2602270033] 2026-02-27 00:33:15 - n2i_forecasting - INFO - Filtered out 1,903 returns
[2602270033] 2026-02-27 00:33:15 - n2i_forecasting - INFO - Prepared 105 weeks | 2023-09-17 → 2025-09-14
[2602270033] 2026-02-27 00:33:15 - n2i_forecasting - INFO - Total revenue: $5,476,537.50
[2602270033] 2026-02-27 00:33:15 - n2i_forecasting - INFO - ------------------------------------------------------------
[2602270033] 2026-02-27 00:33:15 - n2i_forecasting - INFO - CREATING TRAIN/TEST SPL


FORECAST METRICS:
Best model         : ARIMA(1, 1, 1)
MAPE               : 12.21%  → GOOD (< 20.0%) — suitable for strategic / budget planning
MAE                : $5,280.44/week
RMSE               : $7,496.49/week
Revenue trend      : -4.8%  (13wk rolling)
Verdict            : Recent softening vs prior equal window
Note               : Does NOT contradict positive YoY — rolling window compares last 13 weeks to the immediately prior 13 weeks, not the same period last year
Weekly avg revenue : $52,158
Monthly equivalent : $226,792


## 8. Statistical Validation

Every visual pattern from Sections 3–6 subjected to formal hypothesis testing:

| Section | Test | Statistic | p-value | Verdict |
|---|---|---|---|---|
| 3 — Time trend | Spearman | ρ = 0.090 | 0.6743 | ✗ Not significant |
| 4 — Category differences | Kruskal-Wallis | H = 18,977.30 | < 0.001 | ✓ Significant — ε² = 0.550, large effect |
| 5 — Regional differences | Kruskal-Wallis | H = 6.62 | 0.157 | ✗ Not significant |
| 6 — Discount vs. transaction value | Mann-Whitney | U = 140,705,141 | < 0.001 | ✓ Significant — −$7.03, −11.7%, small effect |

**2 of 4 significant.** The two non-significant results are not failures — they are informative negatives. Confirming that time trend and regional differences are not real prevents misallocating resources toward patterns that exist only in the charts.

In [8]:
from n2h_statistical_validation import create_statistical_validation, print_validation_report

# Run statistical validation
# Pass monthly_sales from time_results so the same aggregation is used
# Pass frequency_results from discount_results for complete discount verdict
stats_results = create_statistical_validation(
    df,
    monthly_sales=time_results.get('monthly_sales') if time_results else None,
    frequency_results=discount_results.get('frequency_analysis') if discount_results else None
)

# Summary counts
tests     = stats_results.get('tests', [])
sig_count = sum(1 for t in tests if t.get('significant', False))
print(f"\nStatistical Validation: {sig_count}/{len(tests)} tests significant")

# Full report including verdicts
print_validation_report(stats_results)

[2602270033] 2026-02-27 00:33:16 - n2h_statistical_validation - INFO - ============================================================
[2602270033] 2026-02-27 00:33:16 - n2h_statistical_validation - INFO - STATISTICAL VALIDATION SUITE
[2602270033] 2026-02-27 00:33:16 - n2h_statistical_validation - INFO - ============================================================
[2602270033] 2026-02-27 00:33:16 - n2h_statistical_validation - INFO - TIME TREND VALIDATION:
[2602270033] 2026-02-27 00:33:16 - n2h_statistical_validation - INFO - Testing revenue trend significance (Spearman correlation)...
[2602270033] 2026-02-27 00:33:16 - n2h_statistical_validation - INFO - Spearman ρ=0.090, p=0.6743 — No significant trend detected (p=0.674)
[2602270033] 2026-02-27 00:33:16 - n2h_statistical_validation - INFO - CATEGORY VALIDATION:
[2602270033] 2026-02-27 00:33:16 - n2h_statistical_validation - INFO - Testing category differences (Kruskal-Wallis)...
[2602270033] 2026-02-27 00:33:16 - n2h_statistical_validat


Statistical Validation: 2/4 tests significant


## 9. Executive Dashboard & Summary

Three interactive Plotly dashboards persisted to `outputs/figures/notebook2_fig/`:

| File | Contents |
|---|---|
| `executive_dashboard.html` | 6-panel overview with statistical significance annotations inline |
| `kpi_cards.html` | Headline KPIs: total revenue, avg monthly, peak month, YoY growth |
| `performance_matrix.html` | Category × Region revenue heatmap |

All charts are self-contained HTML — open in any browser without a Python environment.

In [9]:
from n2g_summary_dashboard_enhanced import (
    create_executive_dashboard,
    create_kpi_cards,
    create_performance_matrix,
    generate_analysis_summary,
    print_analysis_summary
)

# ── Compile all results ───────────────────────────────────────────────────────
all_results = {
    'time_trends':          time_results,
    'category':             category_results,
    'region_payment':       region_results,
    'discount':             discount_results,
    'forecasting':          forecast_results,
    'statistical_validation': stats_results,   # <── feeds verdict annotations
}

print("\n" + "=" * 60)
print("CREATING EXECUTIVE DASHBOARDS")
print("=" * 60)

# 1. Executive Sales Dashboard
print("\n1. Executive Sales Dashboard (6-panel + stat annotations)...")
executive_dashboard = create_executive_dashboard(all_results, save=True)
executive_dashboard.show()

# 2. KPI Cards
print("\n2. Key Performance Indicators (4 cards)...")
kpi_cards = create_kpi_cards(all_results, save=True)
kpi_cards.show()

# 3. Performance Matrix
print("\n3. Performance Matrix (Categories + Regions)...")
performance_matrix = create_performance_matrix(all_results, save=True)
performance_matrix.show()

analysis_summary = generate_analysis_summary(all_results)
print_analysis_summary(analysis_summary)

[2602270033] 2026-02-27 00:33:16 - n2g_summary_dashboard_enhanced - INFO - Creating Executive Sales Dashboard...



CREATING EXECUTIVE DASHBOARDS

1. Executive Sales Dashboard (6-panel + stat annotations)...


[2602270033] 2026-02-27 00:33:16 - n2g_summary_dashboard_enhanced - INFO - Saved HTML: executive_dashboard_20260227_003316.html
[2602270033] 2026-02-27 00:33:16 - n2g_summary_dashboard_enhanced - INFO - Executive Sales Dashboard created successfully


[2602270033] 2026-02-27 00:33:16 - n2g_summary_dashboard_enhanced - INFO - Creating KPI Cards...



2. Key Performance Indicators (4 cards)...


[2602270033] 2026-02-27 00:33:16 - n2g_summary_dashboard_enhanced - INFO - Saved HTML: kpi_cards_20260227_003316.html
[2602270033] 2026-02-27 00:33:16 - n2g_summary_dashboard_enhanced - INFO - KPI Cards created successfully


[2602270033] 2026-02-27 00:33:16 - n2g_summary_dashboard_enhanced - INFO - Creating Performance Matrix...
[2602270033] 2026-02-27 00:33:16 - n2g_summary_dashboard_enhanced - INFO - Saved HTML: performance_matrix_20260227_003316.html
[2602270033] 2026-02-27 00:33:16 - n2g_summary_dashboard_enhanced - INFO - Performance Matrix created successfully



3. Performance Matrix (Categories + Regions)...


[2602270033] 2026-02-27 00:33:16 - n2g_summary_dashboard_enhanced - INFO - Generating analysis summary...



                     EXECUTIVE SUMMARY                      

Analysis Timestamp: 2026-02-27 00:33:16

TIME TRENDS:
  Avg monthly revenue: $240,560.20
  Peak month:          2024-12
  Latest YoY growth:   14.67%

CATEGORY:
  Top: Electronics ($3,319,206.50)

FORECASTING:
  Best model: ARIMA(1, 1, 1)
  MAPE: 12.21%
  Trend (rolling): -4.8%



## 10. Results Export

Full `all_results` dictionary serialised to `notebook02_results.pkl`. Preserves complete object fidelity across six keys: `time_trends`, `category`, `region_payment`, `discount`, `forecasting`, `statistical_validation`.

Load the pickle for standalone report regeneration or ad-hoc querying without re-running the full pipeline.

In [10]:
import pickle
from datetime import datetime

# Export all results including stat verdicts 
output_dir = PROJECT_ROOT / 'outputs' / 'analysis_results'
output_dir.mkdir(parents=True, exist_ok=True)

timestamp    = datetime.now().strftime('%Y%m%d_%H%M%S')
results_file = output_dir / f'notebook02_results_{timestamp}.pkl'

with open(results_file, 'wb') as f:
    pickle.dump(all_results, f)

logger.info(f"Results exported to: {results_file}")
print(f"\nResults exported to: {results_file}")
print(f"Keys exported: {list(all_results.keys())}")

[2602270033] 2026-02-27 00:33:16 - __main__ - INFO - Results exported to: c:\Users\Rich Justine Gambe\Documents\AAA Projects\ecommerce-analytics-project\outputs\analysis_results\notebook02_results_20260227_003316.pkl



Results exported to: c:\Users\Rich Justine Gambe\Documents\AAA Projects\ecommerce-analytics-project\outputs\analysis_results\notebook02_results_20260227_003316.pkl
Keys exported: ['time_trends', 'category', 'region_payment', 'discount', 'forecasting', 'statistical_validation']


## 11. Pipeline Complete

All modules executed successfully, results validated, and outputs persisted.

| Module | Key finding |
|---|---|
| Time trends | No significant directional trend (ρ = 0.090, p = 0.67) |
| Category portfolio | Electronics = 56.6% of revenue · ε² = 0.550 large effect confirmed |
| Regional analysis | No significant regional differences — South leads by volume, not value |
| Discount effectiveness | −11.7% per-transaction · +75.1% lifetime revenue uplift |
| ARIMA forecasting | ARIMA(1,1,1) · MAPE = 12.21% · rolling 13-week trend −4.8% |

NB03 reads `rfm_df.parquet` directly from NB01 and does not depend on this notebook. The results pickle is available for downstream report generation or ad-hoc analysis.

In [11]:
# Execution time
notebook_duration = time.time() - NOTEBOOK_START
minutes = int(notebook_duration // 60)
seconds = int(notebook_duration % 60)

logger.info("=" * 60)
logger.info("NOTEBOOK 02 COMPLETED SUCCESSFULLY")
logger.info("=" * 60)
logger.info(f"Execution time: {minutes}m {seconds}s")
logger.info(f"Run ID: {RUN_ID}")
logger.info(f"All figures saved to: {PROJECT_ROOT / config['paths']['notebook2_figures']}")
logger.info("=" * 60)

print("\n" + "=" * 60)
print("NOTEBOOK 02 COMPLETED SUCCESSFULLY")
print("=" * 60)
print(f"Execution time:  {minutes}m {seconds}s")
print(f"Run ID:          {RUN_ID}")
print(f"Figures dir:     {PROJECT_ROOT / config['paths']['notebook2_figures']}")
print("\nDashboards created:")
print("  1. Executive Sales Dashboard (6-panel + statistical annotations)")
print("  2. Key Performance Indicators (4 KPI cards)")
print("  3. Performance Matrix (Category/Region distribution)")
print("=" * 60)

[2602270033] 2026-02-27 00:33:16 - __main__ - INFO - ============================================================
[2602270033] 2026-02-27 00:33:16 - __main__ - INFO - NOTEBOOK 02 COMPLETED SUCCESSFULLY
[2602270033] 2026-02-27 00:33:16 - __main__ - INFO - ============================================================
[2602270033] 2026-02-27 00:33:16 - __main__ - INFO - Execution time: 0m 6s
[2602270033] 2026-02-27 00:33:16 - __main__ - INFO - Run ID: 2602270033
[2602270033] 2026-02-27 00:33:16 - __main__ - INFO - All figures saved to: c:\Users\Rich Justine Gambe\Documents\AAA Projects\ecommerce-analytics-project\outputs\figures\notebook2_fig
[2602270033] 2026-02-27 00:33:16 - __main__ - INFO - ============================================================



NOTEBOOK 02 COMPLETED SUCCESSFULLY
Execution time:  0m 6s
Run ID:          2602270033
Figures dir:     c:\Users\Rich Justine Gambe\Documents\AAA Projects\ecommerce-analytics-project\outputs\figures\notebook2_fig

Dashboards created:
  1. Executive Sales Dashboard (6-panel + statistical annotations)
  2. Key Performance Indicators (4 KPI cards)
  3. Performance Matrix (Category/Region distribution)
